In [27]:
import numpy as np
import pandas as pd

In [28]:
df = pd.read_excel("GSAF5.xls")
df.shape

(7125, 23)

In [29]:
df["Country_original"] = df["Country"]
df["State_original"] = df["State"]   


In [30]:
# Número de valores nulos
print("Nulos en Country: " + str(df["Country"].isna().sum()))
print("Nulos en State: " + str(df["State"].isna().sum()))

# Número de valores diferentes

print("Países diferentes: ", df["Country"].nunique())
print("Estados o regiones diferentes: ", df["State"].nunique())

# Valores más frecuentes

print("\nPaíses:")
print(df["Country"].value_counts(dropna=False).head(30))

print("\nEstados o regiones:")
print(df["State"].value_counts(dropna=False).head(30))



Nulos en Country: 50
Nulos en State: 487
Países diferentes:  257
Estados o regiones diferentes:  956

Países:
Country
USA                 2597
AUSTRALIA           1482
SOUTH AFRICA         597
NEW ZEALAND          144
BAHAMAS              136
PAPUA NEW GUINEA     136
BRAZIL               122
MEXICO               103
ITALY                 72
FIJI                  67
NEW CALEDONIA         65
PHILIPPINES           63
REUNION               60
Australia             56
NaN                   50
EGYPT                 49
CUBA                  49
SPAIN                 46
MOZAMBIQUE            46
INDIA                 41
FRENCH POLYNESIA      37
JAPAN                 36
CROATIA               35
PANAMA                32
JAMAICA               30
SOLOMON ISLANDS       30
IRAN                  29
ENGLAND               25
INDONESIA             24
GREECE                24
Name: count, dtype: int64

Estados o regiones:
State
Florida                  1199
New South Wales           522
NaN                

In [31]:
# Eliminar espacios

df["Country"] = df["Country"].str.strip()
df["State"] = df["State"].str.strip()

# Unificar mayúsculas y minúsculas

df["Country"] = df["Country"].str.upper()
df["State"] = df["State"].str.upper()

In [32]:
# Convertir textos vacíos o marcadores de ausencia de información en nulos

marcadores_nulos = [
"",
"?",
"UNKNOWN",
"UNK",
"N/A",
"NA",
"NONE",
"NULL",
"NOT KNOWN",
"NOT REPORTED"
]

for columna in ["Country", "State"]:
    df[columna] = df[columna].replace(marcadores_nulos, pd.NA)


print("Nulos en Country:", df["Country"].isna().sum())
print("Nulos en State:", df["State"].isna().sum())


Nulos en Country: 50
Nulos en State: 488


In [35]:
# Variantes de nombres de países

correcciones_country = {
"COLUMBIA": "COLOMBIA",
"CEYLON": "SRI LANKA",
"CEYLON (SRI LANKA)": "SRI LANKA",
"MALDIVE ISLANDS": "MALDIVES",
"REUNION ISLAND": "REUNION",
"TURKS AND CAICOS": "TURKS AND CAICOS"
}

df["Country"] = df["Country"].replace(correcciones_country)

correcciones_state = {
    "NSW": "NEW SOUTH WALES",
    "SPLIT-DALMATIA COUNTY": "SPLIT-DALMATIA COUNTY"
}

df["State"] = df["State"].replace(correcciones_state)

# Mostrar todos los nombres de estado
estados = sorted(df["State"].dropna().unique())

for estado in estados:
    print(estado)

"HEAD OF THE GULF"
(SOUTHWESTERN PACIFIC)
04.05N-13.23W
1,000 MILES EAST OF HAWAII
1000 MILES WEST OF HAWAII
10ºS, 142ºE
12 MILES OFF THE NORTH COAST
150 MILES OFFSHORE
165  MILES FROM BERMUDA
180 MILES SOUTHEAST OF OKINAWA
18S / 50E
19S, 178?E
2 TO 3 MILES OFF TABOGUILLA ISLAND, PACIFIC OCEAN
200 NM SOUTHEAST OF MANILA
22ºN, 88ºE
25 KM OFF THE COAST OF IRAN & 483KM FROM MOUTH OF PERSIAN GULF
250 MILES SOUTHWEST OF O'AHU, HAWAII
30 NM FROM SINGAPORE
300 MILES EAST OF LUZON
300 MILES EAST OF MAURITIUS
300 MILES EAST OF ST. THOMAS (VIRGIN ISLANDS)
300 MILES FROM ANTIGUA
330 TO 350 MILES EAST OF WAKE ISLAND
33N, 68W
35º39 : 165º8'
40 MILES OFF GRAND BAHAMA ISLAND
40 MILES SOUTH OF NAPLES
400 MILES SOUTHEAST OF SRI LANKA
5AINT-DENIS
60 MILES NORTH OF SAN DOMINGO IN THE WEST INDIES
600 NM WEST OF THE CANARY ISLANDS
740 MILES SE OF TARAWA ATOLL
800 MILES FROM LAND
9.35N 79.35W
ABACO ISLANDS
ABACOS
ABAU SUB DISTRICT, CENTRAL PROVINCE
ABAU SUBDISTRICT,CENTRAL PROVINCE
ADANA PROVINCE
ADEN
ADMIR

In [36]:
# Detectar posibles ubicaciones que no son países individuales

patron_revision = (r"OCEAN|SEA|BETWEEN|COAST OF|AFRICA|ASIA|" r"BALKANS|/")

df["Country_needs_review"] = (
df["Country"]
.str.contains(patron_revision, case=False, na=False)
)

# Ver qué valores se han marcadoprint(
print(df.loc[df["Country_needs_review"], "Country"].value_counts().to_string())

Country
SOUTH AFRICA                             599
PACIFIC OCEAN                             19
ATLANTIC OCEAN                            17
SOUTH ATLANTIC OCEAN                      12
CARIBBEAN SEA                              8
NORTH PACIFIC OCEAN                        7
INDIAN OCEAN                               7
NORTH ATLANTIC OCEAN                       5
MID ATLANTIC OCEAN                         5
ST HELENA, BRITISH OVERSEAS TERRITORY      2
SOUTH PACIFIC OCEAN                        2
SOUTHWEST PACIFIC OCEAN                    2
MEDITERRANEAN SEA                          2
CORAL SEA                                  1
BRITISH OVERSEAS TERRITORY                 1
ST KITTS / NEVIS                           1
EGYPT / ISRAEL                             1
NORTHERN ARABIAN SEA                       1
SOUTH CHINA SEA                            1
RED SEA                                    1
RED SEA / INDIAN OCEAN                     1
NORTH SEA                                  1
AN

In [39]:
combinaciones = (
    df.groupby(["Country", "State"], dropna=False)
    .size()
    .reset_index(name="Casos")
    .sort_values("Casos", ascending=False)
)

print(combinaciones.head(50).to_string(index=False))

print(
    df.loc[
        df["Country"].isna() & df["State"].notna(),
        ["Country", "State", "Location"]
    ].head(30)
)

for columna in ["Country", "State"]:
    serie = df[columna].dropna().astype(str)
    espacios = serie.str.strip().ne(serie).sum()

    print(f"{columna}: valores con espacios sobrantes = {espacios}")

         Country                        State  Casos
             USA                      FLORIDA   1200
       AUSTRALIA              NEW SOUTH WALES    541
       AUSTRALIA                   QUEENSLAND    357
             USA                       HAWAII    348
             USA                   CALIFORNIA    328
       AUSTRALIA            WESTERN AUSTRALIA    248
    SOUTH AFRICA                KWAZULU-NATAL    219
    SOUTH AFRICA        WESTERN CAPE PROVINCE    197
             USA               SOUTH CAROLINA    176
    SOUTH AFRICA        EASTERN CAPE PROVINCE    168
             USA               NORTH CAROLINA    122
       AUSTRALIA              SOUTH AUSTRALIA    122
       AUSTRALIA                     VICTORIA     98
             USA                        TEXAS     84
          BRAZIL                   PERNAMBUCO     80
     NEW ZEALAND                 NORTH ISLAND     76
       AUSTRALIA                TORRES STRAIT     72
             USA                   NEW JERSEY 

In [40]:
print("Países o ubicaciones diferentes:", df["Country"].nunique())
print("Estados o regiones diferentes:", df["State"].nunique())

print("\nValores más frecuentes de Country:")
print(df["Country"].value_counts(dropna=False).head(30))

print("\nValores más frecuentes de State:")
print(df["State"].value_counts(dropna=False).head(30))

# Comprobar filas completamente duplicadas
print("Filas completamente duplicadas:", df.duplicated().sum())

# Comprobar coincidencias en país, estado y ubicación
print(
    "Filas con misma combinación geográfica:",
    df.duplicated(
        subset=["Country", "State", "Location"],
        keep=False
    ).sum()
)

Países o ubicaciones diferentes: 209
Estados o regiones diferentes: 923

Valores más frecuentes de Country:
Country
USA                 2597
AUSTRALIA           1538
SOUTH AFRICA         599
BAHAMAS              146
NEW ZEALAND          146
PAPUA NEW GUINEA     136
BRAZIL               125
MEXICO               107
ITALY                 72
FIJI                  71
NEW CALEDONIA         68
REUNION               66
PHILIPPINES           65
EGYPT                 53
MOZAMBIQUE            52
NaN                   50
CUBA                  49
SPAIN                 47
INDIA                 41
FRENCH POLYNESIA      40
JAPAN                 36
CROATIA               35
JAMAICA               32
PANAMA                32
SOLOMON ISLANDS       30
IRAN                  29
INDONESIA             25
ENGLAND               25
GREECE                24
HONG KONG             24
Name: count, dtype: int64

Valores más frecuentes de State:
State
FLORIDA                  1200
NEW SOUTH WALES           541
NaN     